# SadTalker Quick Demo (Modern Colab)

This notebook is updated for modern Colab runtimes, current CUDA-enabled PyTorch, and current pip resolver behavior.


## 1) Clone repository

In [ ]:

!git clone -b python312-modernization https://github.com/Inialpha/SadTalker.git
%cd SadTalker

# 2) Install dependencies

In [ ]:

%%bash

pip install -U pip setuptools wheel

pip install torch torchvision torchaudio
pip install opencv-python pillow scipy numpy imageio imageio-ffmpeg pydub tqdm safetensors

pip install facexlib
pip install gfpgan --no-deps
pip install basicsr-fixed

In [ ]:

%%bash

apt-get update -qq
apt-get install -y ffmpeg

## 3) Mount drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4) Download model checkpoints

In [ ]:
from pathlib import Path
import subprocess

REPO_DIR = Path("/content/SadTalker")
print("Downloading checkpoints with scripts/download_models.sh ...")
subprocess.run(["bash", str(REPO_DIR / "scripts" / "download_models.sh")], check=True, cwd=REPO_DIR)
print("Download complete")

## 5) Verify checkpoints

In [ ]:
from pathlib import Path

REPO_DIR = Path("/content/SadTalker")
ckpt = REPO_DIR / "checkpoints"

required = [
    "SadTalker_V0.0.2_256.safetensors",
    "SadTalker_V0.0.2_512.safetensors",
    "mapping_00109-model.pth.tar",
    "mapping_00229-model.pth.tar",
]

missing = [name for name in required if not (ckpt / name).exists()]
if missing:
    raise FileNotFoundError(f"Missing checkpoints: {missing}")

print("Checkpoint verification passed")
for name in required:
    path = ckpt / name
    print(f"- {name}: {path.stat().st_size / (1024**2):.1f} MB")

## 5) Import libraries and verify runtime


In [ ]:
import platform
import sys

import torch
import torchvision
import numpy as np
import scipy
import imageio
import librosa
import gradio as gr

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")
print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("imageio:", imageio.__version__)
print("librosa:", librosa.__version__)
print("gradio:", gr.__version__)

In [ ]:
%xmode verbose

Exception reporting mode: Verbose


In [ ]:
!git pull

## 6) Load models


In [ ]:
import sys
from pathlib import Path

REPO_DIR = Path("/content/SadTalker")
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from src.utils.init_path import init_path
from src.utils.preprocess import CropAndExtract
from src.test_audio2coeff import Audio2Coeff
from src.facerender.animate import AnimateFromCoeff

checkpoint_dir = REPO_DIR / "checkpoints"
config_dir = REPO_DIR / "src" / "config"
device = "cuda" if __import__("torch").cuda.is_available() else "cpu"

paths = init_path(str(checkpoint_dir), str(config_dir), 256, False, "crop")
print("Resolved model paths:")
for k, v in paths.items():
    print(f"- {k}: {v}")

preprocess_model = CropAndExtract(paths, device)
audio_to_coeff = Audio2Coeff(paths, device)
animate_from_coeff = AnimateFromCoeff(paths, device)
print("Model components loaded on", device)

In [ ]:
def generate_video(source_image, driving_audio):

    cmd = [
        "python",
        "inference.py",
        "--source_image", source_image,
        "--driven_audio", driving_audio,
        ...
    ]

    subprocess.run(cmd, check=True)

    return output_video_path

import gradio as gr

def generate_video(source_image, driving_audio):
    """
    For now:
    - source_image is the uploaded image
    - driving_audio is the uploaded audio

    We'll connect this to inference.py next.
    """

    # Temporary placeholder
    return None


with gr.Blocks(title="SadTalker") as demo:

    gr.Markdown("# SadTalker")

    with gr.Row():
        source_image = gr.Image(
            label="Source Image",
            type="filepath"
        )

        driving_audio = gr.Audio(
            label="Driving Audio",
            type="filepath"
        )

    generate_btn = gr.Button("Generate Video", variant="primary")

    output_video = gr.Video(label="Result")

    generate_btn.click(
        fn=generate_video,
        inputs=[
            source_image,
            driving_audio,
        ],
        outputs=output_video,
    )

demo.launch(debug=True)

## 7) Run inference


In [ ]:
from pathlib import Path
import subprocess

REPO_DIR = Path("/content/SadTalker")
image_path = REPO_DIR / "examples" / "source_image" / "full_body_1.png"
audio_path = REPO_DIR / "examples" / "driven_audio" / "bus_chinese.wav"
result_dir = REPO_DIR / "results" / "quick_demo"
result_dir.mkdir(parents=True, exist_ok=True)

cmd = [
    "python", "inference.py",
    "--driven_audio", str(audio_path),
    "--source_image", str(image_path),
    "--result_dir", str(result_dir),
    "--still",
    "--preprocess", "full",
    "--enhancer", "gfpgan",
]

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True, cwd=REPO_DIR)
print("Inference completed")

## 8) Display results


In [ ]:
from pathlib import Path
from IPython.display import Video, display

REPO_DIR = Path("/content/SadTalker")
videos = sorted((REPO_DIR / "results" / "quick_demo").glob("*.mp4"), key=lambda p: p.stat().st_mtime)
if not videos:
    raise FileNotFoundError("No output video found in /content/SadTalker/results/quick_demo")

latest = videos[-1]
print("Latest output:", latest)
display(Video(str(latest), embed=True, width=512))


## 9) Optional Gradio interface


In [ ]:
# Optional: launch the interactive Gradio interface.
# Stop the cell to close the app.

from app_sadtalker import sadtalker_demo

demo = sadtalker_demo(checkpoint_path='checkpoints', config_path='src/config')
demo.queue()
demo.launch(share=False)
